# 第 24 章　分類の評価と閾値

Cammeo らしい米粒を候補に入れ，人がその粒を確認する仮想的な作業を考えます。使うモデルを替えることと，候補に入れる境目を変えることは，確認粒数や見逃しにどう関わるでしょうか。

講義では Book 第 24 章と一緒に，同じ境目で二つのモデルを比べ，次に同じ森で境目を変えます。実習では，自分で目的と二つの閾値を選び，確認する粒数と見逃しを基に判断します。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。ローカルでは配布 ZIP の配置で開くか，ノートと CSV を同じフォルダーに置きます。手元にファイルがなければ公開 URL から CSV を読み込みます。

## 講義　確率を出してから候補を決める

### 三本の木の割合

同じ一粒を三本の木へ入れたとき，それぞれの葉に入った学習粒の Cammeo の割合が次の値だったとします。仕組みを考えるための架空の数です。

| 木 | 葉の Cammeo 数 / 粒数 | 割合 |
|---|---:|---:|
| A | 1 / 5 | 0.20 |
| B | 2 / 5 | 0.40 |
| C | 10 / 10 | 1.00 |

三本の割合を同じ重さで平均してから，0.5 以上なら候補にします。各木で先に 0.5 以上かを決め，多数決にした場合と結論は変わりますか。表の分子と分母をそれぞれ足して割ると，どうなりますか。実際に森を作るときは学習行を復元抽出するため，同じ粒が同じ木に二回引かれた場合，行き先の葉の割合でも二回分に数えます。

### 同じ七列で二つのモデルを学ぶ

Book 第 23 章と同じ米粒の表・学習行・検証行を使います。撮影から得た七つの形状値で，Cammeo（1）か Osmancik（0）かを予測します。検証は前章と同じ 762 粒で，保留はモデルと閾値を選ぶ計算に使いません。

七列ロジスティック回帰は，学習側の標準化と回帰をまとめた `Pipeline` です。森には同じ七列を元の単位で渡します。学習後の `classes_` を見て，どちらの列が Cammeo の確率かを確かめてください。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

file_name = "rice-grains.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

rice = pd.read_csv(csv_path)
features = [
    "area", "perimeter", "major_axis", "minor_axis", "eccentricity",
    "convex_area", "extent",
]
train = rice.loc[rice["split"] == "train"]
valid = rice.loc[rice["split"] == "validation"]

X_train = train[features]
X_valid = valid[features]
y_train = train["variety"].eq("Cammeo").astype(int)
y_valid = valid["variety"].eq("Cammeo").astype(int)
print(f"Training: {len(train):,} grains")
print(f"Validation: {len(valid):,} grains, {y_valid.sum():,} Cammeo")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

shape_model = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(C=1.0, max_iter=2000)),
])
forest = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=2026,
    n_jobs=2,
)

shape_model.fit(X_train, y_train)
forest.fit(X_train, y_train)

print("Logistic regression:", shape_model.classes_)
print("Random forest:      ", forest.classes_)

元行番号 102 の粒を森の 200 本に渡し，木が返す確率と 200 本の平均を見ます。`forest.predict_proba()` が直接返した値と照合します。この値を確認候補に入れるかどうかは，後で閾値を決めてから判断します。

In [ ]:
p_shape = shape_model.predict_proba(X_valid)[:, 1]
p_forest = forest.predict_proba(X_valid)[:, 1]
scored = valid[["record_id", "variety"]].assign(p_shape=p_shape, p_forest=p_forest)

example_row = X_valid.loc[valid["record_id"] == 102]
tree_probabilities = np.array([
    tree.predict_proba(example_row.to_numpy())[0, 1]
    for tree in forest.estimators_
])
p_example = forest.predict_proba(example_row)[0, 1]
print("First five tree probabilities:", tree_probabilities[:5].round(3))
print(f"Mean of {len(tree_probabilities)} trees: {tree_probabilities.mean():.5f}")
print(f"Forest probability: {p_example:.5f}")
print("Same value:", np.isclose(tree_probabilities.mean(), p_example))

### 八粒を四つの件数に振り分ける

下は架空の八粒です。Cammeo の確率が 0.5 **以上**なら候補にします。候補の内外をまず紙などに書いてからセルを実行します。縦に実際の品種，横に候補に入れたかを置いたとき，見逃しと誤警報はどの位置になるかも確認します。

| 粒 | 実際の品種 | Cammeo の予測確率 |
|---|---|---:|
| A | Cammeo | 0.90 |
| B | Osmancik | 0.80 |
| C | Osmancik | 0.44 |
| D | Cammeo | 0.29 |
| E | Cammeo | 0.50 |
| F | Osmancik | 0.12 |
| G | Cammeo | 0.10 |
| H | Osmancik | 0.05 |

In [ ]:
from sklearn.metrics import confusion_matrix

toy_actual = np.array([1, 0, 0, 1, 1, 0, 1, 0])
toy_probability = np.array([0.90, 0.80, 0.44, 0.29, 0.50, 0.12, 0.10, 0.05])
toy_prediction = (toy_probability >= 0.5).astype(int)
toy_matrix = confusion_matrix(toy_actual, toy_prediction, labels=[0, 1])

pd.DataFrame(toy_matrix, index=["Actual 0", "Actual 1"], columns=["Decision 0", "Decision 1"])

右下の TP は，実際に Cammeo で，候補に入れた粒です。上の行が Osmancik，下の行が Cammeo，右の列が確認候補です。次の三つを手で計算し，比率によって条件にする行・列がどう違うかを読みます。

- **正解率**は，正しく振り分けた TN と TP を全粒数で割ります。
- **適合率**は，候補の中で実際に Cammeo だった割合です。
- **再現率**は，実際の Cammeo のうち候補に入った割合です。

見逃した Cammeo は，再現率の分母に数えます。適合率の分母に数えるのは，候補に入れた粒です。

### 同じ 0.5 でモデルを比べる

`evaluate_threshold()` は，渡された確率に閾値を適用し，混同行列の四件数と割合を返します。`TN, FP, FN, TP` のうち，見逃しは `FN`，候補に入った別品種は `FP` です。最初はどちらのモデルも同じ 0.5 で判定し，四件数を並べます。森へ替えると見逃しと誤警報がどう変わるかを，両方の図の同じ位置で読みます。

元行番号 1632 の粒の確率も照合します。この粒を，二つの方法はそれぞれ候補に入れるでしょうか。表示を二桁に丸めた確率で先に判定すると何が変わるかも考えてください。

In [ ]:
def evaluate_threshold(actual, probability, threshold):
    decision = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, decision, labels=[0, 1]).ravel()
    candidates = tp + fp
    positive = tp + fn
    return {
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "candidates": candidates,
        "accuracy": (tn + tp) / len(actual),
        "precision": tp / candidates if candidates else np.nan,
        "recall": tp / positive if positive else np.nan,
    }

In [ ]:
at_half = pd.DataFrame(
    [
        evaluate_threshold(y_valid, p_shape, 0.5),
        evaluate_threshold(y_valid, p_forest, 0.5),
    ],
    index=["Logistic regression", "Random forest"],
)
count_columns = ["TN", "FP", "FN", "TP"]
at_half[count_columns]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.5), sharey=True, layout="constrained")
for ax, (model_name, counts) in zip(axes, at_half.iterrows()):
    matrix = counts[count_columns].to_numpy().reshape(2, 2)
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=800)
    for row in range(2):
        for column in range(2):
            count = matrix[row, column]
            ax.text(column, row, f"{count:.0f}", ha="center", va="center",
                    color="#182a36", fontsize=14)
    ax.set(title=model_name, xlabel="Decision",
           xticks=[0, 1], xticklabels=["Outside", "Candidate"],
           yticks=[0, 1], yticklabels=["Osmancik", "Cammeo"])
axes[0].set_ylabel("Actual variety")
plt.show()

In [ ]:
scored.loc[scored["record_id"] == 1632].round(5)

### 同じ森で 0.5 と 0.2 を比べる

次の図の二本の線の間は，新しく候補へ入る確率の範囲です。上段の帯では，どの種類の誤りが減りますか。下段の帯では，どの種類が増えるでしょうか。帯だけを取り出し，実際の品種を数えます。次に，同じ森の確率に二つの閾値を渡し，四件数と候補数，三つの割合を照合します。最後の表の単位は % です。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7.2, 5.1), sharex=True, sharey=True, layout="constrained")
bins = np.linspace(0, 1, 21)
for ax, actual, name, color in [
    (axes[0], 1, "Actual Cammeo (326 grains)", "#286b70"),
    (axes[1], 0, "Actual Osmancik (436 grains)", "#a34f30"),
]:
    ax.axvspan(0.2, 0.5, color="#f6e5bc", label="New candidates")
    ax.hist(p_forest[y_valid == actual], bins=bins, color=color, edgecolor="white")
    ax.axvline(0.2, color="#687581", linewidth=1.2, linestyle=":")
    ax.axvline(0.5, color="#34434c", linewidth=1.2, linestyle="--")
    ax.set(title=name, ylabel="Grains", xlim=(0, 1))
axes[0].legend(frameon=False, loc="upper left")
axes[-1].set(xlabel="Cammeo probability from the forest", xticks=[0, 0.2, 0.5, 0.8, 1])
plt.show()

In [ ]:
new_candidates = scored.loc[(scored["p_forest"] >= 0.2) & (scored["p_forest"] < 0.5)]
new_candidates["variety"].value_counts().reindex(["Cammeo", "Osmancik"])

In [ ]:
thresholds_to_compare = [0.5, 0.2]
forest_by_threshold = pd.DataFrame(
    [evaluate_threshold(y_valid, p_forest, threshold) for threshold in thresholds_to_compare],
    index=pd.Index(thresholds_to_compare, name="Threshold"),
)
forest_by_threshold[count_columns + ["candidates"]]

In [ ]:
(forest_by_threshold[["accuracy", "precision", "recall"]] * 100).round(2)

## 実習　目的に合わせて閾値を選ぶ

### 1. 優先することと二つの閾値を選ぶ（約 5 分）

Cammeo の見逃しを抑えたいか，人が確認する候補を抑えたいか，先に決めます。そのために試したい二つの閾値を 0〜1 の間から選び，少なくとも片方は講義の 0.5・0.2 以外にします。例えば 0.1，0.35，0.6 も試せます。

低い方の閾値にすると新しく候補に入るのは，どこからどこまでの確率でしょうか。候補数と見逃しはどちら向きに変わり得るか，境目を根拠に予想します。

（優先すること，選んだ閾値と理由，予想を記入）

### 2. 同じ確率で二つの規則を試す（約 9 分）

`lower_threshold` と `higher_threshold` の `None` を選んだ数に替えます。四件数と候補数の表，適合率・再現率（%）の表，新しく候補に入った品種の件数が表示されます。二つの規則で `p_forest` は共通です。

低い方に替えると，候補は何粒増え，見逃しは何粒減りましたか。新たに入った Cammeo と Osmancik の数を使い，どの二つの誤りがどう動いたかを説明します。

In [ ]:
lower_threshold = None
higher_threshold = None

if lower_threshold is None or higher_threshold is None:
    print("低い方と高い方の閾値を指定します。")
elif not 0 <= lower_threshold < higher_threshold <= 1:
    print("0〜1 の範囲で，低い方から順に二つの異なる値を指定します。")
else:
    chosen_thresholds = [higher_threshold, lower_threshold]
    comparison = pd.DataFrame(
        [evaluate_threshold(y_valid, p_forest, threshold) for threshold in chosen_thresholds],
        index=pd.Index(chosen_thresholds, name="Threshold"),
    )
    display(comparison[count_columns + ["candidates"]])
    display((comparison[["precision", "recall"]] * 100).round(2))

    added = scored.loc[(scored["p_forest"] >= lower_threshold)
                       & (scored["p_forest"] < higher_threshold)]
    counts = added["variety"].value_counts().reindex(["Cammeo", "Osmancik"], fill_value=0)
    print(f"New candidates: {lower_threshold} <= p < {higher_threshold}")
    display(counts.rename("grains").to_frame())

### 3. 選ぶ理由を報告する（約 6 分）

確認する人に，次の内容を 3〜4 文で伝えます。

1. 何を優先したかと，二つの閾値のうちどちらを選ぶかを書く。確認候補の粒数と Cammeo の見逃しを，両方の閾値について挙げる。
2. もう一方の閾値を選ぶ理由は，どんな場合に生じるかを，追加した候補の品種や，適合率・再現率の違いから説明する。
3. 実際に別の粒へ使う前にどの試料や撮影条件でもう一度数えたいかを書く。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。目的と予想，選んだ二つの閾値の表，報告を残したノートブックを提出します。

### データの出典

Cinar, I., & Koklu, M. (2019). *Rice (Cammeo and Osmancik)*. UCI Machine Learning Repository. 教材用 CSV は列名を短縮し，元の行番号と授業用の分割を追加しています。 [データ配布元](https://doi.org/10.24432/C5MW4Z) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) ・ [原著論文](https://doi.org/10.18201/ijisae.2019355381)